In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
df=pd.read_csv('/kaggle/input/old-car-price-prediction/car_price.csv')


# Data cleansing:

**Duplicated values:**

In [ ]:
df[df.duplicated()].count()

In [ ]:
df.info()

**NA values:**

In [ ]:
df[df.isna()].count()

In [ ]:
df=df.drop(columns='Unnamed: 0',axis=1)

In [ ]:
df.loc[df['car_prices_in_rupee'].str.contains('Lakh'), 'Currency'] = 'Lakh'
df.loc[df['car_prices_in_rupee'].str.contains('Crore'), 'Currency'] = 'Crore'

In [ ]:
df=df.dropna()


In [ ]:
df.head()

**Dealing with Currencies and other objects:**

In [ ]:
df['kms_driven']=df['kms_driven'].str.replace(' kms','')
df['car_prices_in_rupee']=df['car_prices_in_rupee'].str.replace(' Lakh','')
df['kms_driven']=df['kms_driven'].str.replace(',','')
df['Seats']=df['Seats'].str.replace(' Seats','')
df['engine']=df['engine'].str.replace(' cc','')
df['car_prices_in_rupee']=df['car_prices_in_rupee'].str.replace(' Crore','')


In [ ]:
df['car_prices_in_rupee']=df['car_prices_in_rupee'].astype('float')
df['kms_driven']=df['kms_driven'].astype('int')
df['Seats']=df['Seats'].astype('int')
df['engine']=df['engine'].astype('int')

**Currency exchange:**

In [ ]:

df['car_prices_in_rupee']=np.where(df['Currency'] == 'Crore',
                                           df['car_prices_in_rupee'] * 100,
                                           df['car_prices_in_rupee'])

In [ ]:
df['car_prices_in_rupee']=df['car_prices_in_rupee']*100000

In [ ]:
df2=pd.read_csv('/kaggle/input/old-car-price-prediction/car_price.csv')
df2=df2[df2["car_prices_in_rupee"].str.contains("Lakh")==False]
df2=df2[df2["car_prices_in_rupee"].str.contains("Crore")==False]
df2['kms_driven']=df2['kms_driven'].str.replace(' kms','')
df2['kms_driven']=df2['kms_driven'].str.replace(',','')
df2['Seats']=df2['Seats'].str.replace(' Seats','')
df2['engine']=df2['engine'].str.replace(' cc','')
df2['car_prices_in_rupee']=df2['car_prices_in_rupee'].str.replace(',','')
df2['car_prices_in_rupee']=df2['car_prices_in_rupee'].astype('int')
df2['kms_driven']=df2['kms_driven'].astype('int')
df2['Seats']=df2['Seats'].astype('int')
df2['engine']=df2['engine'].astype('int')

In [ ]:
df2.info()

In [ ]:
df3=[df,df2]
df=pd.concat(df3)

In [ ]:
df=df.drop(columns='Unnamed: 0',axis=1)
df=df.drop(columns='Currency',axis=1)

In [ ]:
df.info()

**Now we have a clean dataset:**

# Data analysis:

In [ ]:
df.head()

In [ ]:
plt.style.use('Solarize_Light2')

In [ ]:
pv1=pd.pivot_table(df, index=['manufacture'],values = ['car_prices_in_rupee'],aggfunc = 'mean') 
pv1.plot(kind='line',linewidth=4.5,figsize=(12,7),title='Average car price by Year')

**The average car price has been moving up over years**

In [ ]:
pv2=pd.pivot_table(df, index=['Seats'],values = ['car_prices_in_rupee'],aggfunc = 'mean') 
pv2.plot(kind='bar',figsize=(12,7),title='Average car price by Number of seats',edgecolor = 'black')

**Cars with 4 seats has the highest average price in the market**

In [ ]:
plt.style.use('Solarize_Light2')
plt.figure(figsize=(12,7))
plt.title('Relation between price and KMS driven')
plt.scatter(df.kms_driven,df.car_prices_in_rupee,color="b")


**Of course, more KMS means a lower price**

In [ ]:
plt.style.use('Solarize_Light2')

In [ ]:
plt.style.use('Solarize_Light2')
price=df['car_prices_in_rupee']
price.plot(kind='hist',figsize=(12,7),bins=80,edgecolor = 'black',title='Price Distribution')



**Most of the cars price range between 0 and 5000000 rupee**

In [ ]:
top5=pd.pivot_table(df,index='car_name',values='car_prices_in_rupee').sort_values(by='car_prices_in_rupee',ascending=False)
top5=top5.head(5)
top5.plot(kind='barh',figsize=(10,7),edgecolor = 'black',title='The Most expensive 5 cars')

**Most 5 expensive cars in the market**

In [ ]:
own=pd.pivot_table(df,index='ownership',values='car_prices_in_rupee')
own.plot(kind='line',figsize=(10,7),linewidth=4.5,title='Price level by number of owners')

**More owners mean a lower price **

In [ ]:
eng=pd.pivot_table(df,index='manufacture',values='engine',aggfunc='mean')
eng.plot(kind='line',figsize=(10,7),linewidth=4.5,title='Engines developement over years')

**In general, the average engine rate is increasing over years**

In [ ]:
transm=pd.pivot_table(df,index='transmission',values='car_prices_in_rupee',columns='fuel_type')
transm.plot(kind='barh',figsize=(10,8),edgecolor = 'black',title='Average price by Fuel type')

**Automatic Diesel cars has the highest average price is the market, while Manuel Lpg got the lowest average car price**

In [ ]:
fuelc=pd.pivot_table(df,index='fuel_type',values='car_name',aggfunc='count').sort_values(by='car_name')
fuelc.plot(kind='barh', edgecolor = 'black',figsize=(10,8),title='Number of cars by Fuel type')

**Petrol fuel cars are the most common fuel type in the market**

In [ ]:
plt.style.use('Solarize_Light2')
transmc=pd.pivot_table(df,index='transmission',values='car_name',aggfunc='count')
transmc.plot(kind='pie',figsize=(10,8),subplots=True,autopct='%1.1f%%',title='Transmission type popularity')

**Manuel transission cars are more common**

In [ ]:
train=df.copy()

In [ ]:
train['car_name']=pd.factorize(train.car_name)[0]
train['fuel_type']=pd.factorize(train.fuel_type)[0]
train['transmission']=pd.factorize(train.transmission)[0]
train['ownership']=pd.factorize(train.ownership)[0]

In [ ]:
train.corr()

In [ ]:
plt.style.use('Solarize_Light2')
sns.heatmap(train.corr(),annot = True, cmap= 'YlGnBu', fmt= '.2f')

**As we can see, transmission type, manufacture year, engine rate, fuel type and Kms driven has a strong impact on the car price.**

# Machine learning part:

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn import metrics
from sklearn.metrics import r2_score

In [ ]:
train.info()

**Setting Features and terget**

In [ ]:
X= train.drop(columns='car_prices_in_rupee',axis=1)
Y= train['car_prices_in_rupee']

**Splitting the dataset**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.15, random_state = 100)

**Model training:**

In [ ]:
xgbr = xgb.XGBRegressor(verbosity=0)
xgbr.fit(X_train,y_train)

**Model accuracy:**

In [ ]:
score1 = xgbr.score(X_train,y_train)
score1

In [ ]:
y_pred1 = xgbr.predict(X_test)
rscore1=r2_score(y_test, y_pred1)
rscore1

**We have %74 as score for our test data, it's not that good, Can you suggest some techniques to increase that accuracy?**

In [ ]:
df['Price_prediction']=xgbr.predict(X)

In [ ]:
df.head(15)

In [ ]:
prd=df['Price_prediction']
x_ax = range(len(Y))
plt.figure(figsize=(12,7))
plt.plot(x_ax, Y,linewidth = '4.5', label="Original")
plt.plot(x_ax, prd, linewidth = '4.5', label="predicted")
plt.title("Test data VS Predicted data")
plt.xlabel('X')
plt.ylabel('Y')
plt.legend(loc='lower right',fancybox=True, shadow=True)
plt.show() 

In [ ]:

plt.scatter(df['Price_prediction'],df['car_prices_in_rupee'],color="b")
plt.title('Prediction and Original data correlation')

**We can see that Prediction and Origianl data has a high correlation around  the 0-1000000 rupee range, this model can be updated in the future after trying some hypertuning parameter to increase the accuracy.
If you like this notebook, don't forget to upvote it! Thank you!**